# Vectora — run 3c v2: deep mode + judge export (the last GPU piece of run 3)

**v2:** v1 (2026-10-26) ran, but its labelling file listed one source per paper, unnumbered, so claims cited [2], [4] could not be checked. Fixed in `bcafb2d`; this version deletes any stale copy of the code, checks that the fix is present, and writes to a `run3c_v2_*` folder.

Run 3b's deep-mode job died partway (no error in the log; most likely killed) and, because `vectora answers` only saved at the end, lost everything. That is fixed: the report is now saved every 10 questions, so a second kill keeps what was scored.

| # | Experiment | GPU time |
|---|---|---|
| D | **deep** mode, first 80 questions + 20 negatives: grounding judge on (judged by qwen2.5:7b), answers by gemma3:4b | ~40–60 min |
| F | export 30 judged answers for hand labelling | seconds |

Plus ~15 min of setup. Uses the new defaults (BGE-small, MMR off, measured floors, reranker pool 8).

**Before you leave it:** Runtime → Change runtime type → **T4 GPU**. Run **cell 1 alone** and approve Google Drive.
Check cell 2 prints a GPU line (not "no compatible GPU"). Then Runtime → **Run all**, and leave the tab open with the laptop awake and plugged in.

Each experiment saves to `MyDrive/vectora_results/<run>/` as it finishes, so a disconnect loses at most the one in progress. 

## 1. Connect Drive (needs you awake)

In [ ]:
import datetime, os
RUN = "run3c_v2_" + datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
from google.colab import drive
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/vectora_results/{RUN}"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 2. Ollama on the GPU

In [ ]:
ANSWER = "gemma3:4b"     # run 2's best answer model
JUDGE  = "qwen2.5:7b"    # utility + judge: a different model, so it does not grade itself
MODELS = [ANSWER, JUDGE]

# pciutils + lshw let the installer detect the GPU; without them it installs a CPU-only build.
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_NUM_PARALLEL": "4",
                               "OLLAMA_MAX_LOADED_MODELS": "2", "OLLAMA_CONTEXT_LENGTH": "8192"})
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2); print("ollama up"); break
    except Exception:
        time.sleep(1)
for m in MODELS:
    !ollama pull {m} 2>&1 | tail -1
!ollama run {ANSWER} "hi" > /dev/null 2>&1; ollama ps
!grep -i -E "inference compute|no compatible gpu" /content/ollama.log | head -3

## 3. Vectora + config

In [ ]:
!rm -rf /content/Vectora   # a stale copy from an earlier session would silently keep the old code
!git clone -q https://github.com/SiddharthaRaja/Vectora.git /content/Vectora
%cd /content/Vectora
!git log --oneline -1 | tee {OUT}/commit.txt
import subprocess as _sp
assert "source_chunk_ids" in open("evaluation/answers.py").read(), "old code: the labelling fix is missing; stop and re-clone"
print("labelling fix present:", _sp.run(["git", "merge-base", "--is-ancestor", "bcafb2d", "HEAD"]).returncode == 0)
!pip install -q -e ".[ingest,eval]" 2>&1 | grep -v -i "warning\|dependency resolver\|incompatible" | tail -2
open(".env", "w").write(f"""
OPENAI_API_KEY=ollama
OPENAI_BASE_URL=http://127.0.0.1:11434/v1
VECTORA_ANSWER_MODEL={ANSWER}
VECTORA_UTILITY_MODEL={JUDGE}
VECTORA_DAILY_COST_CEILING_USD=0
VECTORA_LLM_TIMEOUT_SECONDS=180
VECTORA_LLM_MAX_RETRIES=2
VECTORA_CORPUS=qasper-300
VECTORA_QDRANT_URL=
VECTORA_QDRANT_PATH=./data/qdrant
VECTORA_CORS_ORIGINS=http://localhost:5000
VECTORA_LOG_LEVEL=WARNING
""")

## 4. Build the 300-paper corpus, ingest, index

Same command as on your laptop, so the same 300 papers and 365 title-free questions (the golden set is also in git; `--force` rebuilds it identically).
Every case has the annotators' evidence paragraphs and all their answers.

In [ ]:
!python scripts/qasper_corpus.py --corpus qasper-300 --split all --papers 300 --force 2>&1 | tail -6
!git diff --stat corpora/qasper-300/golden.yaml | tail -1 | tee {OUT}/golden_diff.txt   # empty = identical to the committed set
!date; vectora ingest 2>&1 | tail -3
!vectora index --recreate 2>&1 | tail -3; date
!cp corpora/qasper-300/golden.yaml {OUT}/golden.yaml

## 8. Experiment D: `deep` mode with the faithfulness judge

Deep mode runs the grounding check (judged by `qwen2.5:7b`) and the relevance judge. First 80 questions + 20 negatives, roughly 40–60 minutes. Saved every 10 questions.

In [ ]:
!date
!free -h | head -2
!PYTHONUNBUFFERED=1 vectora answers --mode deep --limit 80 --output {OUT}/expD_deep.json 2>&1 | tee {OUT}/expD_deep.log | tail -45
!date

## 10. Experiment F: export 30 answers for hand labelling

Writes `judge_labels.csv` to your Drive folder. Later, on your laptop: open it, **hide the `judge_*` columns**, and for each row put `1` in `human_faithful` if every claim in the answer is supported by the sources shown, `0` if any claim is not.
Then send it to me and I'll compute agreement.

In [ ]:
!python -m evaluation.judge_agreement export --answers {OUT}/expD_deep.json --chunks corpora/qasper-300/chunks.jsonl --out {OUT}/judge_labels.csv --n 30

## 11. Wrap up

In [ ]:
!cp corpora/qasper-300/reports/* {OUT}/ 2>/dev/null; cp /content/ollama.log {OUT}/ 2>/dev/null; date | tee {OUT}/finished_at.txt
!ls -la {OUT}